# Wide-Sigma CMA-ES on the Scenario-Aware Base — a Tuned Profile, Submitted

**This agent is [pilkwang's Scenario-Aware Economic Policy](https://www.kaggle.com/code/pilkwang/kaggriculture-scenario-aware-economic-policy)
at heart — copied with admiration — plus a machine-tuned knob layer on top.**
Our thanks go first to the original author. This notebook submits the tuned profile so
the score is public and reproducible; the method is documented below so you can tune
your own.

## The one-line story

Narrow searches around the original's hand-set parameters converge in ~15 generations —
*into the original's basin* (it is remarkably close to a local optimum; respect). A
**wider initial step (sigma 0.35, popsize 24)** escapes that basin and finds a
different profile that measurably beats the replay-tape agent family while staying
even with the original. A further 100-generation continuation found nothing better —
we believe the parameter frontier of this architecture is now closed, and said so
publicly before submitting this.

## What changed vs the original

Eleven hand-set scalars lifted into a knob dict, then searched by CMA-ES
(fitness = mean money edge + worst-opponent edge + win rate, against a frozen pool;
per-game parallel evaluation; rotating seeds; held-out validation — training fitness
carries ~5x winner's-curse inflation, never trust it raw).

| Knob | Original | This profile | Meaning |
|---|---:|---:|---|
| `reserve_mult` | 1.00 | 0.95 | slightly lower reservation prices — sell a touch earlier |
| `forecast_guard` | 0.88 | 0.85 | weaker hold-for-tomorrow guard |
| `glide_left` | 12 | 14.2 | liquidation glide starts ~2 days earlier |
| `mid_herd` / `target_herd` | 11 / 15 | 10.9 / 15.9 | herd staging |
| `herd_expansion_day` / `final` | 7 / 11 | 6.5 / 10.7 | slightly earlier |
| `melon_base` / `max` | 10 / 12 | 10.2 / 11.7 | melon acreage |
| `max_hands` | 12 | 12.8 | labor ceiling |
| `feed_stock_days` | 3 | 2.1 | leaner feed buffer |

Held-out results (both seats, fresh seeds): **23/32 (72%) vs the Anchor-V13
replay-tape family** (including a probe-router variant), ~50% vs the original
scenario-aware agent, and a large margin vs pre-notebook-era agents. The interesting
frontier above this baseline is structural, not parametric.

In [ ]:
# Pin the engine: this agent follows the kaggle-environments 1.32.2
# execution contract (same pin as the original notebook).
import subprocess, sys
from importlib import metadata
try:
    ver = metadata.version("kaggle-environments")
except metadata.PackageNotFoundError:
    ver = "0"
if tuple(int(x) for x in ver.split(".")[:2]) < (1, 32):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "kaggle-environments==1.32.2"], check=True)
import kaggle_environments as ke
print("kaggle-environments", ke.__version__)

In [ ]:
# Assemble the agent (base64 keeps 115KB of quote-heavy source notebook-safe).
import base64
AGENT_B64 = r'''...'''
src = base64.b64decode(AGENT_B64).decode()
with open("main.py", "w") as f:
    f.write(src)
print("main.py written:", len(src), "chars")

In [ ]:
# Smoke test: one full game against the built-in baseline.
from kaggle_environments import make
env = make("kaggriculture", configuration={"seed": 5150}, debug=True)
env.run(["main.py", "starter"])
final = env.steps[-1]
print("final money:", [s.reward for s in final])
assert (final[0].reward or 0) > 100000, "agent underperforming — check assembly" 

In [ ]:
# Package for submission.
import tarfile
with tarfile.open("submission.tar.gz", "w:gz") as tar:
    tar.add("main.py")
print("submission.tar.gz ready")

## Credits

- **[pilkwang](https://www.kaggle.com/pilkwang)** — the entire agent architecture and
  economics. Our sincere thanks;
  [the original is here](https://www.kaggle.com/code/pilkwang/kaggriculture-scenario-aware-economic-policy).
- The Anchor V13 / replay-tape lineage authors, whose public agents made a measurable
  tuning target.
- Method notes and an earlier profile:
  [our tribute notebook](https://www.kaggle.com/code/pengwang91/tuned-knob-layer-on-scenario-aware-tribute).

*Copied with respect, tuned with machines, submitted in the open.*